```yaml
title: 神经网络分类通用架构
subtitle: 基于CNN-MINST
date: 20240322
```

特点: 

- logger
- same_seed
- train,valid,test
- config

In [ ]:
import math
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
# 防止torch包与Anaconda环境中的同一个文件出现了冲突，画不出图
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import logging

## logger,seed

In [ ]:
def same_seed(seed):
    '''固定seed保证复现'''
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_logger():
    logger = logging.getLogger(__name__)
    logging.basicConfig(filename='example.log',
                        encoding='utf-8',
                        level=logging.INFO)
    work_dir = f'./logs/{time.strftime("%Y-%m-%d", time.localtime())}'
    if not os.path.exists(work_dir):
        os.makedirs(work_dir)
    logger_time = time.strftime("%Y-%m-%d-%H.%M", time.localtime())

    log_file = os.path.join(work_dir, f'{logger_time}.log')
    logging.basicConfig(level=logging.INFO
        format="[%(asctime)s][%(filename)s][line:%(lineno)d][%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d-%H.%M",
        handlers=[
            logging.FileHandler(log_file, mode='w'),
            logging.StreamHandler()
        ]
        )
    return logger

## dataset
MNIST do not need special data loader

In [ ]:
class myDataset(Dataset):

    def __init__(self):
        ...

    def __len__(self):
        ...

    def __getitem__(self, index):
        ...

## dataloader and split


In [ ]:
def get_dataloader(data_dir, batch_size: int, n_workers: int):
    """Generate dataloader"""
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = datasets.MNIST(root='./',
                             train=True,
                             download=True,
                             transform=transform)

    # Split dataset into training dataset and validation dataset
    trainlen = int(0.9 * len(dataset))
    lengths = [trainlen, len(dataset) - trainlen]
    trainset, validset = random_split(dataset, lengths)

    train_loader = DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=n_workers,
        pin_memory=True,
    )
    valid_loader = DataLoader(
        validset,
        batch_size=batch_size,
        num_workers=n_workers,
        drop_last=True,
        pin_memory=True,
    )

    return train_loader, valid_loader

## model


In [ ]:
class MyModel(nn.Module):
    '''定义模型'''

    def __init__(self):
        super(MyModel, self).__init__()
        self.conv = nn.Sequential(  # (1,28,28)
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=(5, 5)),
            nn.ReLU(),  # (6,24,24)
            nn.MaxPool2d((2, 2)),  # (6,12,12)
            nn.Conv2d(6, 12, 3),
            nn.ReLU(),  # (12,10,10)
            nn.MaxPool2d((2, 2))  # (12,5,5)
        )

        self.fc = nn.Sequential(nn.Linear(12 * 5 * 5, 128), nn.ReLU(),
                                nn.Linear(128, 32), nn.ReLU(),
                                nn.Linear(32, 10))
        self.net = nn.Sequential(  # (1,28,28)
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=(5, 5)),
            nn.ReLU(),  # (6,24,24)
            nn.MaxPool2d((2, 2)),  # (6,12,12)
            nn.Conv2d(6, 12, 3),
            nn.ReLU(),  # (12,10,10)
            nn.MaxPool2d((2, 2)),  # (12,5,5)
            nn.Flatten(),
            nn.Linear(12 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 10))

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


## training model

In [ ]:
def train(model, train_loader, criterion, optimizer, device: str, epoch: int):
    model.train()
    train_loop = tqdm(train_loader, position=0, leave=False)
    train_loss = 0
    for iter, (data, target) in enumerate(train_loop):
        data, target = data.to(device), target.to(device)
        output = model(data)
        # targets的类型是要求long(int64)，这里对齐
        loss = criterion(output, target.long())
        # 清零梯度，反向传播，更新权重
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # 进度条设置
        train_loop.set_description(f'Epoch {epoch}')
        train_loop.set_postfix({'loss': loss.item()})
        train_loss += loss.item()
    train_loss = train_loss / len(train_loader.dataset)
    return train_loss


def valid(model, valid_loader, criterion, device: str, epoch: int):
    model.eval()
    epoch_preds = []
    # TODO: epoch_labels累加是否等于valid_dataset？在不打乱情况下
    epoch_labels = []
    correct = 0
    valid_loop = tqdm(valid_loader,
                      position=0,
                      ncols=70,
                      leave=False,
                      desc=f'Epoch {epoch} validating')
    with torch.no_grad():
        for data, target in valid_loop:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target.long())
            test_loss += loss.item()

            pred = output.argmax(dim=1)
            correct += pred.eq(target.view_as(pred)).sum().item()
            epoch_preds.extend(pred.cpu().data.numpy())
            epoch_labels.extend(target.cpu().data.numpy())

    test_loss = test_loss / len(valid_loader.dataset)
    accuracy = correct / len(valid_loader.dataset)

    return test_loss, accuracy, epoch_preds, epoch_labels

In [ ]:
def trainer(train_loader, valid_loader, model, config):
    #===prepare===
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    stats = {
        "best_loss": math.inf,
        "train_loss": [],
        "valid_loss": [],
        "valid_pred": None,
        "early_stop_count": 0,
    }

    for epoch in range(config.n_epoches):
        #===train mode===
        model.train()
        train_loss = train(model, train_loader, criterion, optimizer,
                           config.device, epoch)
        logger.info(f"Train | Epoch: {epoch}\t Loss: {train_loss:.2e}")
        stats['train_loss'].append(train_loss)

        #===evaluate mode===
        test_loss, accuracy, epoch_preds, epoch_labels = valid(
            model, valid_loader, criterion, config.device, epoch)

        print(f'{epoch_labels==valid_loader.dataset}1111111111')
        logger.info(
            f"Valid | Epoch: {epoch}\t Loss: {train_loss:.2e}\t Accuracy: {accuracy:.2f}\t F1 Score:{score:.2f}"
        )

        record_valid_loss.append(np.mean(loss_record))

        #===early stopping===
        if record_valid_loss[-1] < best_loss:
            best_loss = record_valid_loss[-1]
            print(
                f'Now model with loss {best_loss:4f} valid accuracy {accuracy:4f}... from epoch {epoch}'
            )
            early_stop_count = 0
        else:
            early_stop_count += 1
        if early_stop_count >= config.early_stop:
            print(
                f'Model is not improving for {config.early_stop} epoches. The last epoch is {epoch}.'
            )
            break
    # save_path=config.save(config.time+f'model_{loss:.3f}.ckpt')
    torch.save(model.state_dict(), config.save_model(best_loss))
    print(f'Saving model with loss {best_loss:4f}... from epoch {epoch}')
    return record_train_loss, record_valid_loss, best_loss


## test(if need)


In [ ]:
# test_data = datasets.MNIST(root='./',
#                            train=False,
#                            download=True,
#                            transform=transform)

## main function and config setting

In [ ]:
class config:
    '''超参数设定，用`print(pd.DataFrame([config.__dict__]))`查看当前参数'''

    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.seed = 45
        self.batch_size = 512
        self.valid_ratio = 0.1
        self.folder = 'run'
        # 路径名不能出现冒号
        self.time = time.strftime(r"%Y-%m-%d_%H.%M_", time.localtime())
        #-==Important Hyperparameters===
        self.early_stop = 5
        self.learning_rate = 10e-3
        self.n_epoches = 100

    def save(self, path: str):
        if not os.path.exists(self.folder):
            os.makedirs(self.folder)
        return os.path.join(self.folder, self.time + path)

    def save_model(self, loss, accuracy=None):
        if accuracy == None:
            path = f'loss{loss:4f}_model.ckpt'
        else:
            path = f'accuracy{accuracy:3f}_model.ckpt'
        return self.save(path)


if __name__ == '__main__':
    config = config()
    same_seed(config.seed)
    print(f'{torch.__version__=}\n{config.device=}')
    # print(config.device)

    #===data processing===
    train_loader, valid_loader = get_dataloader(data_dir='./')

    #===training===
    model = MyModel().to(config.device)
    # print(model)
    global logger
    logger = get_logger()

    train_loss, valid_loss, best_loss = trainer(train_loader, valid_loader,
                                                model)

    #===plot loss===
    loss_plot(train_loss, valid_loss)

    #===predict===
    model = MyModel().to(config.device)
    model.load_state_dict(
        torch.load(config.save_model(best_loss), map_location=config.device))
    # 使用之前的model迁移学习
    # model.load_state_dict(torch.load(r'.\run\2023-04-18_22.38_epoch1000_score0.989000_model.ckpt',map_location=config.device),strict=False)
    preds, accuracy, incorrect_index = predict(test_data, model)
    print(f'test accuracy:{accuracy:4f}')
    os.rename(config.save_model(best_loss),
              config.save_model(best_loss, accuracy))

    #===confusion_matrix===
    cm = confusion_matrix(test_data.targets.numpy(),
                          preds,
                          labels=[i for i in range(10)])
    cm_plot(cm, accuracy)
    print(cm)

    #===incorrect comparasion===
    incorrect_plot(test_data, preds, incorrect_index)
    print('===FINISH!===')